In [38]:
from __future__ import annotations

from dataclasses import dataclass
import numpy as np
import pandas as pd


MINUTES_IN_YEAR = 525_600
N30 = 30 * 24 * 60  # 43,200 minutes


def _require(cond: bool, msg: str) -> None:
    if not cond:
        raise ValueError(msg)


@dataclass(frozen=True)
class PreparedChain:
    """
    Clean boundary object: a single-expiration option chain prepared for VIX math.

    - Calls/Puts are indexed by strike with columns: bid, ask, mid.
    - strikes is the sorted union of strikes across both sides.
    """
    calls: pd.DataFrame
    puts: pd.DataFrame
    strikes: np.ndarray


def validate_chain_for_vix(chain: pd.DataFrame) -> None:
    """
    Optional validation for your upstream pipeline.

    This checks only "structural" assumptions so the calculation stays deterministic:
      - required columns exist
      - cp_flag is {C,P}
      - no duplicates per (cp_flag, strike)
      - finite/nonnegative quotes and bid <= ask
      - strikes finite and > 0

    The official methodology notes that series with null quotes or bid>ask are not candidates
    (and K0 null/bid>ask makes the index not calculable), so we treat those as invalid here. :contentReference[oaicite:1]{index=1}
    """
    required = {"strike", "cp_flag", "bid", "ask"}
    missing = required - set(chain.columns)
    _require(not missing, f"Missing required columns: {missing}")

    df = chain[list(required)].copy()
    df["cp_flag"] = df["cp_flag"].astype(str).str.upper()

    _require(df["cp_flag"].isin(["C", "P"]).all(), "cp_flag must be 'C' or 'P'.")
    _require(np.isfinite(df["strike"]).all() and (df["strike"] > 0).all(), "strike must be finite and > 0.")
    _require(np.isfinite(df["bid"]).all() and (df["bid"] >= 0).all(), "bid must be finite and >= 0.")
    _require(np.isfinite(df["ask"]).all() and (df["ask"] >= 0).all(), "ask must be finite and >= 0.")
    _require((df["bid"] <= df["ask"]).all(), "Found bid > ask (invalid quotes).")

    dup = df.duplicated(subset=["cp_flag", "strike"], keep=False)
    _require(not dup.any(), "Duplicate rows for (cp_flag, strike); deduplicate upstream.")


def prepare_chain_for_vix(chain: pd.DataFrame) -> PreparedChain:
    """
    Minimal preparation consistent with the official methodology:

      - Remove series with null quotes (explicit step in strike selection). :contentReference[oaicite:2]{index=2}
      - Remove series where bid > ask (officially not candidates for ATM and can make index incalculable). :contentReference[oaicite:3]{index=3}
      - Compute mid = (bid + ask)/2
      - Split into calls/puts indexed by strike (uniqueness enforced via verify_integrity=True)

    No "extra" sanity checks beyond what’s needed to make the downstream calculation deterministic.
    """
    cols = ["strike", "cp_flag", "bid", "ask"]
    missing = set(cols) - set(chain.columns)
    _require(not missing, f"Missing required columns: {missing}")

    df = chain.loc[:, cols].copy()
    df["cp_flag"] = df["cp_flag"].astype(str).str.upper()
    df = df[df["cp_flag"].isin(["C", "P"])]

    # Official step: remove null quotes. :contentReference[oaicite:4]{index=4}
    df = df.dropna(subset=["bid", "ask", "strike"])

    # Exclude invalid markets (bid > ask) per the official candidate rules. :contentReference[oaicite:5]{index=5}
    df = df[df["bid"] <= df["ask"]]

    df["mid"] = 0.5 * (df["bid"] + df["ask"])

    calls = (
        df[df["cp_flag"] == "C"]
        .set_index("strike", verify_integrity=True)[["bid", "ask", "mid"]]
        .sort_index()
    )
    puts = (
        df[df["cp_flag"] == "P"]
        .set_index("strike", verify_integrity=True)[["bid", "ask", "mid"]]
        .sort_index()
    )

    strikes = np.sort(df["strike"].unique())
    return PreparedChain(calls=calls, puts=puts, strikes=strikes)


@dataclass(frozen=True)
class TermResult:
    minutes_to_expiry: int
    T: float
    R: float
    K_atm: float
    F: float
    K0: float
    sigma2: float
    sum_piece: float
    qk: pd.DataFrame  # columns: strike, Q


def _select_K_atm_and_forward(prep: PreparedChain, R: float, T: float) -> tuple[float, float]:
    """
    Cboe step (Forward Price and K0):
      - K_atm is the strike minimizing |C_mid - P_mid| (tie -> lowest strike)
      - F = K + exp(RT) * (C_mid - P_mid)

    Note: the math methodology excludes series with null quotes or bid>ask from ATM candidacy;
    prepare_chain_for_vix() already filtered those. :contentReference[oaicite:6]{index=6}
    """
    common = prep.calls.join(prep.puts, how="inner", lsuffix="_C", rsuffix="_P")
    _require(len(common) > 0, "No strikes with both call and put quotes; cannot infer forward.")

    tmp = pd.DataFrame(
        {
            "strike": common.index.to_numpy(dtype=float),
            "abs_diff": (common["mid_C"] - common["mid_P"]).abs().to_numpy(),
        }
    ).sort_values(["abs_diff", "strike"], ascending=[True, True])

    K_atm = float(tmp.iloc[0]["strike"])
    row = common.loc[K_atm]
    F = float(K_atm + np.exp(R * T) * (row["mid_C"] - row["mid_P"]))
    return K_atm, F


def _strike_at_or_below(strikes: np.ndarray, x: float) -> float:
    leq = strikes[strikes <= x]
    _require(len(leq) > 0, "No strike <= forward; check strike grid / forward inference.")
    return float(leq[-1])


def _select_wing_two_consecutive_zero_bid_or_ask(wing: pd.DataFrame, *, direction: str) -> pd.DataFrame:
    """
    Cboe strike selection (current language):

      - Exclude any OTM option with bid == 0 OR ask == 0
      - Once two consecutive strikes are found with zero bid OR zero ask,
        exclude those observed options and consider no strikes beyond.

    This reflects the post–Feb 10, 2025 update in the official math methodology. :contentReference[oaicite:7]{index=7}
    """
    if wing.empty:
        return wing

    ordered = wing.sort_index(ascending=(direction == "up"))
    is_zero = (ordered["bid"].to_numpy() == 0) | (ordered["ask"].to_numpy() == 0)

    cut = None
    for i in range(len(is_zero) - 1):
        if is_zero[i] and is_zero[i + 1]:
            cut = i
            break

    if cut is None:
        # Keep all non-zero quotes
        return ordered.loc[~pd.Series(is_zero, index=ordered.index)]
    else:
        # Stop strictly before the first zero in the consecutive pair; also exclude any zero rows before the cut.
        kept = ordered.iloc[:cut]
        kept_zero = is_zero[:cut]
        return kept.loc[~pd.Series(kept_zero, index=kept.index)]


def _build_qk_table(prep: PreparedChain, K0: float) -> pd.DataFrame:
    """
    Cboe strike selection + pricing Q(K):

      - K < K0: OTM puts selected by the zero-quote rule
      - K > K0: OTM calls selected by the zero-quote rule
      - K = K0: include BOTH put and call; Q(K0) is their average midquote

    If all OTM puts or all OTM calls are excluded, the index cannot be calculated. :contentReference[oaicite:8]{index=8}
    """
    _require(K0 in prep.calls.index and K0 in prep.puts.index, "K0 must have both a call and a put quote.")

    puts_otm = prep.puts.loc[prep.puts.index < K0]
    calls_otm = prep.calls.loc[prep.calls.index > K0]

    puts_sel = _select_wing_two_consecutive_zero_bid_or_ask(puts_otm, direction="down")
    calls_sel = _select_wing_two_consecutive_zero_bid_or_ask(calls_otm, direction="up")

    _require(len(puts_sel) > 0, "All OTM puts excluded; volatility index cannot be calculated.")
    _require(len(calls_sel) > 0, "All OTM calls excluded; volatility index cannot be calculated.")

    q0 = float(0.5 * (prep.calls.loc[K0, "mid"] + prep.puts.loc[K0, "mid"]))

    qk = pd.concat(
        [
            pd.DataFrame({"strike": puts_sel.index.to_numpy(dtype=float), "Q": puts_sel["mid"].to_numpy(dtype=float)}),
            pd.DataFrame({"strike": np.array([K0], dtype=float), "Q": np.array([q0], dtype=float)}),
            pd.DataFrame({"strike": calls_sel.index.to_numpy(dtype=float), "Q": calls_sel["mid"].to_numpy(dtype=float)}),
        ],
        ignore_index=True,
    )

    qk = qk.groupby("strike", as_index=False)["Q"].mean().sort_values("strike").reset_index(drop=True)
    return qk


def _deltaK(strikes: np.ndarray) -> np.ndarray:
    """
    Cboe ΔK definition:
      - endpoints: adjacent difference
      - interior: (K_{i+1} - K_{i-1}) / 2
    """
    strikes = np.asarray(strikes, dtype=float)
    _require(len(strikes) >= 2, "Need at least 2 included strikes to compute ΔK.")

    dK = np.empty_like(strikes)
    dK[0] = strikes[1] - strikes[0]
    dK[-1] = strikes[-1] - strikes[-2]
    dK[1:-1] = 0.5 * (strikes[2:] - strikes[:-2])
    return dK


def compute_single_term_variance(prep: PreparedChain, *, R: float, minutes_to_expiry: int) -> TermResult:
    """
    Cboe single-term variance for one expiration.
    """
    _require(minutes_to_expiry > 0, "minutes_to_expiry must be positive.")
    T = minutes_to_expiry / MINUTES_IN_YEAR

    K_atm, F = _select_K_atm_and_forward(prep, R=R, T=T)
    K0 = _strike_at_or_below(prep.strikes, F)

    qk = _build_qk_table(prep, K0=K0)

    strikes = qk["strike"].to_numpy(dtype=float)
    Q = qk["Q"].to_numpy(dtype=float)
    dK = _deltaK(strikes)

    disc = np.exp(R * T)
    sum_piece = float(np.sum((dK / (strikes ** 2)) * disc * Q))

    # σ^2 = (2/T)*Σ[...] - (1/T)*(F/K0 - 1)^2
    sigma2 = float((2.0 / T) * sum_piece - (1.0 / T) * ((F / K0) - 1.0) ** 2)

    return TermResult(
        minutes_to_expiry=minutes_to_expiry,
        T=T,
        R=R,
        K_atm=K_atm,
        F=F,
        K0=K0,
        sigma2=sigma2,
        sum_piece=sum_piece,
        qk=qk,
    )

def interpolate_vix_30day(near: TermResult, next_: TermResult, target_minutes: int = N30) -> float:
    """
    Cboe constant-maturity interpolation to 30 days (in minutes), then VIX = 100*sqrt(var_30).
    """
    N1, N2 = near.minutes_to_expiry, next_.minutes_to_expiry
    _require(N1 < target_minutes < N2, "Need N1 < target_minutes < N2 for interpolation.")

    w1 = (N2 - target_minutes) / (N2 - N1)
    w2 = (target_minutes - N1) / (N2 - N1)

    var_30 = (w1 * near.T * near.sigma2 + w2 * next_.T * next_.sigma2) * (MINUTES_IN_YEAR / target_minutes)
    return float(100.0 * np.sqrt(var_30))

In [39]:
import numpy as np
import pandas as pd
import wrds
from zoneinfo import ZoneInfo

SPX_SECID = 108105
NY = ZoneInfo("America/New_York")

def _table_for_date(dt: pd.Timestamp) -> str:
    return f"optionm_all.opprcd{dt.year}"

def fetch_opprcd_slice(db: wrds.Connection, date: str,
                       exdate_min_days: int = 20,
                       exdate_max_days: int = 45) -> pd.DataFrame:
    d = pd.Timestamp(date)
    table = _table_for_date(d)
    ex_min = (d + pd.Timedelta(days=exdate_min_days)).date()
    ex_max = (d + pd.Timedelta(days=exdate_max_days)).date()

    sql = f"""
        SELECT
            secid, date, exdate, cp_flag, strike_price, best_bid, best_offer,
            am_settlement, ss_flag, contract_size, expiry_indicator
        FROM {table}
        WHERE secid = {SPX_SECID}
          AND date = '{d.date()}'
          AND exdate BETWEEN '{ex_min}' AND '{ex_max}'
          AND contract_size = 100
          AND ss_flag = '0'
          AND cp_flag IN ('C','P')
          AND best_bid IS NOT NULL
          AND best_offer IS NOT NULL
          AND strike_price IS NOT NULL
          AND best_bid <= best_offer
    """
    return db.raw_sql(sql)

def fetch_zero_curve(db: wrds.Connection, date: str) -> pd.DataFrame:
    d = pd.Timestamp(date).date()
    sql = f"""
        SELECT date, days, rate
        FROM optionm_all.zerocd
        WHERE date = '{d}'
        ORDER BY days
    """
    return db.raw_sql(sql)

def minutes_to_expiry(asof_date: str, exdate: pd.Timestamp, am_settlement: float) -> int:
    asof = pd.Timestamp(asof_date).tz_localize(NY) + pd.Timedelta(hours=16)
    exp = (
        pd.Timestamp(exdate).tz_localize(NY)
        + (pd.Timedelta(hours=9, minutes=30) if int(am_settlement) == 1 else pd.Timedelta(hours=16))
    )
    return int((exp - asof).total_seconds() // 60)

def pick_near_next_terms(df_day: pd.DataFrame, asof_date: str, target_minutes: int):
    df = df_day.copy()
    df["exdate"] = pd.to_datetime(df["exdate"])

    # VIX universe
    am = df["am_settlement"].fillna(0).astype(int)
    is_am = am == 1

    # Exclude PM expiries that coincide with AM expiries (Cboe rule)
    am_expiries = set(df.loc[is_am, "exdate"].dt.date.unique())
    is_same_date_pm = (am == 0) & (df["exdate"].dt.date.isin(am_expiries))

    # PM-settled weeklies => keep max exdate within each week (W-FRI) ---
    pm_weekly = (am == 0) & (df["expiry_indicator"] == "w")

    # Define "week" as ending on Friday; if Friday is a holiday, the max exdate is typically Thu.
    wk = df["exdate"].dt.to_period("W-FRI")

    is_pm_eow = pd.Series(False, index=df.index)
    if pm_weekly.any():
        max_ex = df.loc[pm_weekly].groupby(wk[pm_weekly])["exdate"].transform("max")
        is_pm_eow.loc[pm_weekly] = df.loc[pm_weekly, "exdate"].eq(max_ex).to_numpy()

    # Keep AM (SPX) + PM EOW weeklies, drop PM that share exdate with AM
    df = df[(is_am | is_pm_eow) & (~is_same_date_pm)]

    # Pick Near/Next terms bracketing 30 days ---
    series = df[["exdate", "am_settlement"]].drop_duplicates().copy()
    series["N"] = [
        minutes_to_expiry(asof_date, ex, am_)
        for ex, am_ in zip(series["exdate"], series["am_settlement"])
    ]
    series = series[series["N"] > 0].sort_values("N")

    near = series[series["N"] < target_minutes].tail(1)
    nxt  = series[series["N"] > target_minutes].head(1)
    if near.empty or nxt.empty:
        raise ValueError("Could not find Near/Next terms bracketing 30 days in the pulled expiry window.")

    e1 = (pd.Timestamp(near.iloc[0]["exdate"]), int(near.iloc[0]["am_settlement"]))
    e2 = (pd.Timestamp(nxt.iloc[0]["exdate"]), int(nxt.iloc[0]["am_settlement"]))
    return e1, e2

def interp_rate_from_zerocd(zc: pd.DataFrame, days_to_exp: float) -> float:
    """
    Linearly interpolate the OptionMetrics zero curve by days.
    Note: `rate` is continuously-compounded, but appears to be in *percent* units on WRDS for your sample.
    """
    x = zc["days"].to_numpy(dtype=float)
    y = zc["rate"].to_numpy(dtype=float)
    return float(np.interp(days_to_exp, x, y))

def rate_to_decimal(r: float) -> float:
    return r / 100.0 if r > 1.0 else r

def build_chain(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "strike": df["strike_price"].astype(float) / 1000.0,
        "cp_flag": df["cp_flag"].astype(str),
        "bid": df["best_bid"].astype(float),
        "ask": df["best_offer"].astype(float),
    })

In [40]:
# asof_date = "2025-03-10"
# target_minutes = 30 * 24 * 60

# db = wrds.Connection()

# df = fetch_opprcd_slice(db, asof_date, exdate_min_days=20, exdate_max_days=45)
# zc = fetch_zero_curve(db, asof_date)

# (ex1, am1), (ex2, am2) = pick_near_next_terms(df, asof_date, target_minutes)

# df1 = df[(pd.to_datetime(df["exdate"]) == ex1) & (df["am_settlement"].fillna(0).astype(int) == am1)]
# df2 = df[(pd.to_datetime(df["exdate"]) == ex2) & (df["am_settlement"].fillna(0).astype(int) == am2)]

# chain1 = build_chain(df1)
# chain2 = build_chain(df2)

# N1 = minutes_to_expiry(asof_date, ex1, am1)
# N2 = minutes_to_expiry(asof_date, ex2, am2)

# d = pd.Timestamp(asof_date)
# days1 = N1 / (24*60)
# days2 = N2 / (24*60)

# R1 = rate_to_decimal(interp_rate_from_zerocd(zc, days1))
# R2 = rate_to_decimal(interp_rate_from_zerocd(zc, days2))

# prep1 = prepare_chain_for_vix(chain1)
# prep2 = prepare_chain_for_vix(chain2)

# near = compute_single_term_variance(prep1, R=R1, minutes_to_expiry=N1)
# nxt  = compute_single_term_variance(prep2, R=R2, minutes_to_expiry=N2)

# vix = interpolate_vix_30day(near, nxt, target_minutes=target_minutes)
# print("VIX:", vix)

In [41]:
from zoneinfo import ZoneInfo

NY = ZoneInfo("America/New_York")
SPX_SECID = 108105
N30 = 30 * 24 * 60
MINUTES_IN_YEAR = 525_600

def opprcd_table_for_year(y: int) -> str:
    return f"optionm_all.opprcd{y}"

def fetch_expiry_calendar(db, start_date: str, end_date: str, secid: int) -> pd.DataFrame:
    # Buffer expiries so near/next exist; you can tighten later
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    ex_min = (start + pd.Timedelta(days=10)).date()
    ex_max = (end + pd.Timedelta(days=60)).date()

    table = opprcd_table_for_year(start.year)
    if start.year != end.year:
        raise NotImplementedError("PoC assumes one year; extend by looping years.")

    sql = f"""
    SELECT DISTINCT
        date, exdate, am_settlement
    FROM {table}
    WHERE secid = {secid}
      AND date BETWEEN '{start.date()}' AND '{end.date()}'
      AND exdate BETWEEN '{ex_min}' AND '{ex_max}'
      AND contract_size = 100
      AND ss_flag = '0'
    """
    cal = db.raw_sql(sql)
    cal["date"] = pd.to_datetime(cal["date"])
    cal["exdate"] = pd.to_datetime(cal["exdate"])
    cal["am_settlement"] = cal["am_settlement"].fillna(0).astype(int)
    return cal

In [42]:
def minutes_to_expiry(asof_date: pd.Timestamp, exdate: pd.Timestamp, am_settlement: int) -> int:
    asof = asof_date.tz_localize(NY) + pd.Timedelta(hours=16)
    exp = exdate.tz_localize(NY) + (pd.Timedelta(hours=9, minutes=30) if am_settlement == 1 else pd.Timedelta(hours=16))
    return int((exp - asof).total_seconds() // 60)

def build_term_schedule(cal: pd.DataFrame, target_minutes: int = N30) -> pd.DataFrame:
    out = []
    for d, g in cal.groupby("date", sort=True):
        # Universe filter:
        am_expiries = set(g.loc[g["am_settlement"] == 1, "exdate"].dt.date.unique())
        is_am = g["am_settlement"] == 1
        is_pm_eow = (g["am_settlement"] == 0) & (g["exdate"].dt.weekday == 4)  # Friday
        is_same_date_pm = (g["am_settlement"] == 0) & (g["exdate"].dt.date.isin(am_expiries))

        gg = g[(is_am | is_pm_eow) & (~is_same_date_pm)].copy()
        if gg.empty:
            continue

        gg["N"] = gg.apply(lambda r: minutes_to_expiry(d, r["exdate"], int(r["am_settlement"])), axis=1)
        gg = gg[gg["N"] > 0].sort_values("N")

        near = gg[gg["N"] < target_minutes].tail(1)
        nxt  = gg[gg["N"] > target_minutes].head(1)
        if near.empty or nxt.empty:
            continue

        out.append({"date": d, "term": "near", "exdate": near.iloc[0]["exdate"], "am_settlement": int(near.iloc[0]["am_settlement"]), "N": int(near.iloc[0]["N"])})
        out.append({"date": d, "term": "next", "exdate":  nxt.iloc[0]["exdate"],  "am_settlement": int(nxt.iloc[0]["am_settlement"]),  "N": int(nxt.iloc[0]["N"])})
    return pd.DataFrame(out)

In [43]:
def _values_clause(pairs: pd.DataFrame) -> str:
    rows = []
    for r in pairs.itertuples(index=False):
        d = pd.Timestamp(r.date).date().isoformat()
        e = pd.Timestamp(r.exdate).date().isoformat()
        am = int(r.am_settlement)
        # Typed date literals => Postgres infers correct DATE type
        rows.append(f"(DATE '{d}', DATE '{e}', {am})")
    return ",\n".join(rows)

def fetch_option_quotes_for_schedule(db, schedule: pd.DataFrame, secid: int) -> pd.DataFrame:
    y = schedule["date"].dt.year.unique()
    if len(y) != 1:
        raise NotImplementedError("PoC assumes one year; extend by looping years.")
    table = f"optionm_all.opprcd{int(y[0])}"

    pairs = schedule[["date", "exdate", "am_settlement"]].drop_duplicates()
    values_sql = _values_clause(pairs)

    sql = f"""
    WITH needed(date, exdate, am_settlement) AS (
        VALUES
        {values_sql}
    )
    SELECT
        o.date, o.exdate, o.am_settlement, o.cp_flag,
        o.strike_price, o.best_bid, o.best_offer
    FROM {table} o
    JOIN needed n
      ON o.date = n.date
     AND o.exdate = n.exdate
     AND COALESCE(o.am_settlement, 0)::int = n.am_settlement
    WHERE o.secid = {secid}
      AND o.contract_size = 100
      AND o.ss_flag = '0'
      AND o.cp_flag IN ('C','P')
      AND o.best_bid IS NOT NULL
      AND o.best_offer IS NOT NULL
      AND o.best_bid <= o.best_offer
      AND o.strike_price IS NOT NULL
    """
    df = db.raw_sql(sql)
    df["date"] = pd.to_datetime(df["date"])
    df["exdate"] = pd.to_datetime(df["exdate"])
    df["am_settlement"] = df["am_settlement"].fillna(0).astype(int)
    return df

In [44]:
def fetch_zerocd_range(db, start_date: str, end_date: str) -> pd.DataFrame:
    sql = f"""
    SELECT date, days, rate
    FROM optionm_all.zerocd
    WHERE date BETWEEN '{pd.Timestamp(start_date).date()}' AND '{pd.Timestamp(end_date).date()}'
    ORDER BY date, days
    """
    zc = db.raw_sql(sql)
    zc["date"] = pd.to_datetime(zc["date"])
    return zc

In [45]:
def _interp_rate(zc_day: pd.DataFrame, days_to_exp: float) -> float:
    x = zc_day["days"].to_numpy(float)
    y = zc_day["rate"].to_numpy(float)
    return float(np.interp(days_to_exp, x, y))

def compute_vix_series(schedule: pd.DataFrame, quotes: pd.DataFrame, zerocd: pd.DataFrame):
    # Index lookups for speed
    qg = quotes.groupby(["date", "exdate", "am_settlement"], sort=False)
    zg = {d: g for d, g in zerocd.groupby("date", sort=False)}

    rows = []
    for d, dd in schedule.groupby("date", sort=True):
        try:
            near_row = dd[dd["term"] == "near"].iloc[0]
            next_row = dd[dd["term"] == "next"].iloc[0]

            def build_chain(date, exdate, am):
                df = qg.get_group((date, exdate, am))
                return pd.DataFrame({
                    "strike": df["strike_price"].astype(float) / 1000.0,
                    "cp_flag": df["cp_flag"].astype(str),
                    "bid": df["best_bid"].astype(float),
                    "ask": df["best_offer"].astype(float),
                })

            chain1 = build_chain(d, near_row["exdate"], int(near_row["am_settlement"]))
            chain2 = build_chain(d, next_row["exdate"], int(next_row["am_settlement"]))

            zc_day = zg.get(d)
            if zc_day is None:
                raise ValueError("Missing zerocd for date")

            days1 = (near_row["exdate"].normalize() - d.normalize()).days
            days2 = (next_row["exdate"].normalize() - d.normalize()).days
            R1 = _interp_rate(zc_day, days1) / 100.0
            R2 = _interp_rate(zc_day, days2) / 100.0

            prep1 = prepare_chain_for_vix(chain1)
            prep2 = prepare_chain_for_vix(chain2)

            near = compute_single_term_variance(prep1, R=R1, minutes_to_expiry=int(near_row["N"]))
            nxt  = compute_single_term_variance(prep2, R=R2, minutes_to_expiry=int(next_row["N"]))

            vix = interpolate_vix_30day(near, nxt, target_minutes=N30)

            rows.append({
                "date": d.date(),
                "vix": vix,
                "near_exdate": near_row["exdate"].date(),
                "next_exdate": next_row["exdate"].date(),
                "R1": R1, "R2": R2,
                "N1": int(near_row["N"]), "N2": int(next_row["N"]),
            })
        except Exception as e:
            rows.append({"date": d.date(), "vix": np.nan, "error": str(e)})

    return pd.DataFrame(rows).sort_values("date")

In [46]:
start_date = "2025-03-01"
end_date = "2025-03-31"   

db = wrds.Connection()

cal = fetch_expiry_calendar(db, start_date, end_date, SPX_SECID)
schedule = build_term_schedule(cal, target_minutes=N30)

quotes = fetch_option_quotes_for_schedule(db, schedule, SPX_SECID)
zc = fetch_zerocd_range(db, start_date, end_date)

vix_df = compute_vix_series(schedule, quotes, zc)

print(vix_df)

Loading library list...
Done
          date        vix near_exdate next_exdate        R1        R2     N1  \
0   2025-03-03  22.943641  2025-03-28  2025-04-04  0.044455  0.044522  35940   
1   2025-03-04  23.954366  2025-03-28  2025-04-04  0.044397  0.044463  34500   
2   2025-03-05  21.936858  2025-04-04  2025-04-11  0.044469  0.044527  43140   
3   2025-03-06  25.263556  2025-04-04  2025-04-11  0.044504  0.044557  41700   
4   2025-03-07  23.284475  2025-04-04  2025-04-11  0.044579  0.044627  40260   
5   2025-03-10  28.058041  2025-04-04  2025-04-11  0.044534  0.044580  36000   
6   2025-03-11  27.108946  2025-04-04  2025-04-11  0.044416  0.044463  34560   
7   2025-03-12  23.842324  2025-04-04  2025-04-17  0.044377  0.044457  33120   
8   2025-03-13  25.086463  2025-04-11  2025-04-17  0.044402  0.044435  41760   
9   2025-03-14  21.536816  2025-04-11  2025-04-17  0.044427  0.044457  40320   
10  2025-03-17  20.537406  2025-04-11  2025-04-17  0.044445  0.044474  36000   
11  2025-03

In [47]:
schedule.head(10), schedule.tail(10), quotes.groupby(['date','exdate','am_settlement']).size().head()

(        date  term     exdate  am_settlement      N
 0 2025-03-03  near 2025-03-28              0  35940
 1 2025-03-03  next 2025-04-04              0  46020
 2 2025-03-04  near 2025-03-28              0  34500
 3 2025-03-04  next 2025-04-04              0  44580
 4 2025-03-05  near 2025-04-04              0  43140
 5 2025-03-05  next 2025-04-11              0  53220
 6 2025-03-06  near 2025-04-04              0  41700
 7 2025-03-06  next 2025-04-11              0  51780
 8 2025-03-07  near 2025-04-04              0  40260
 9 2025-03-07  next 2025-04-11              0  50340,
          date  term     exdate  am_settlement      N
 32 2025-03-25  near 2025-04-17              1  32730
 33 2025-03-25  next 2025-04-25              0  44640
 34 2025-03-26  near 2025-04-17              1  31290
 35 2025-03-26  next 2025-05-02              0  53280
 36 2025-03-27  near 2025-04-25              0  41760
 37 2025-03-27  next 2025-05-02              0  51840
 38 2025-03-28  near 2025-04-25       